# Challenge — Final Hidden-Test Predictions

The locked system from Sections 2 & 3:
- **HistGradientBoostingClassifier** per class (best config from the grid),
- **per-class thresholds** tuned on local validation,
- **median filter** with the best window from Section 3.

We generate predictions for all hidden-test files and save them in the
required `filename, annotation, onset, offset` format.

## 1 — Setup

In [ ]:
import os, sys, glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PHASE = Path('/Users/amrabdullah/AI_2026ss/mlpc/phases/04_challenge')
sys.path.insert(0, str(PHASE))
import utils

ART = PHASE / 'artifacts'
SUB = PHASE / 'submissions'

hgb = joblib.load(ART / 'section2_histgb_best.joblib')
thr = np.load(ART / 'section2_thresholds.npy')
with open(ART / 'section3_summary.json') as f:
    best_w = int(json.load(f)['best_window'])
print(f'HistGB + thresholds + median window={best_w} loaded')

hidden_files = sorted(glob.glob(str(utils.PATH_TEST / 'audio_features' / '*.npz')))
print(f'hidden test files: {len(hidden_files)}')

## 2 — Generate hidden-test predictions

In [ ]:
t0 = time.time()
pred_hidden = utils.generate_predictions(hidden_files, hgb, thresholds=thr,
                                          median_window=best_w, use_probs=True)
print(f'  generated {len(pred_hidden)} predicted intervals in {time.time()-t0:.0f}s')
print(f'  unique files predicted: {pred_hidden["filename"].nunique()}')
print(f'  classes predicted: {sorted(pred_hidden["annotation"].unique())}')

# Sanity check on format
required_cols = ['filename', 'annotation', 'onset', 'offset']
assert all(c in pred_hidden.columns for c in required_cols), 'missing required columns'
assert (pred_hidden['offset'] > pred_hidden['onset']).all(), 'offset must be > onset'
assert pred_hidden['annotation'].isin(utils.CLASS_NAMES).all(), 'unknown class in predictions'
print('format check OK')

OUTPUT = SUB / 'predictions_hidden_test.csv'
pred_hidden.to_csv(OUTPUT, index=False)
print(f'saved -> {OUTPUT}')
print(f'rows: {len(pred_hidden)}')

pred_hidden.head(10)

## 3 — Quick distribution check (vs baseline submission)

In [ ]:
baseline_csv = PHASE / 'baseline' / 'predictions_hidden_test.csv'
if baseline_csv.is_file():
    baseline_df = pd.read_csv(baseline_csv)
    print(f'baseline submission rows: {len(baseline_df)}')
    print(f'ours: {len(pred_hidden)}  delta: {len(pred_hidden) - len(baseline_df):+d}')
    by_cls = pd.DataFrame({
        'baseline_n_events': baseline_df.groupby('annotation').size(),
        'ours_n_events':     pred_hidden.groupby('annotation').size(),
    }).reindex(utils.CLASS_NAMES).fillna(0).astype(int)
    by_cls['delta'] = by_cls['ours_n_events'] - by_cls['baseline_n_events']
    print('\nclass-wise event count (hidden test):')
    print(by_cls.to_string())
    by_cls.to_csv(ART / 'final_class_event_counts.csv')